In [ ]:
from pathlib import Path
import re, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import trapezoid
from scipy.optimize import least_squares, curve_fit, minimize_scalar

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})

# ---- paths -------------------------------------------------------------------------
DATA_DIR = Path(".")                       # folder holding the client .xlsx workbooks
OUT = Path("step_analysis_outputs"); OUT.mkdir(exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)

# Workbooks with no raw step time series (see measurement inventory)
EXCLUDE_WORKBOOKS = ("AY21", "AUG18")
# Sheets that contain step-like tests but are out of scope for step-up/down
EXCLUDE_SHEETS = ("bureau",)               # FEB18 'Bureau Step Up & Stop' = start-up from rest

# ---- step verification ---------------------------------------------------------------
RATE_TOL = 0.05            # measured initial/final rate must be within 5 % of the protocol label
RATE_SETTLE_TOL = 0.05     # rate is 'settled' once it stays within 5 % of the target (or 0.01 1/s)
MIN_INSTRUMENT_WINDOW = 0.05   # s; never analyse stress earlier than this after the step (motor inertia)

# ---- fixed transient / stable rule -----------------------------------------------------
# Justification: the structural time constant of blood is tau_lambda ~ 1.1 s (Frontiers 2022, 12 donors).
#   transient window  W_T = [instrument settle, 2 s]   (about 2 tau_lambda: structure still evolving)
#   stable window     W_S = [30 s, end of test]        (> 25 tau_lambda: structure should have equilibrated)
# The gap 2-30 s is 'late transient' and belongs to neither label.
TRANSIENT_END = 2.0
STABLE_START = 30.0
STABLE_MAX_DRIFT = 0.10    # W_S is a *defensible* stable state only if stress drifts < 10 % across it
SETTLE_BAND = 0.10         # settling time: stress stays within +-10 % of the stable value
NOISE_PA = 0.001           # Pa; absolute noise floor (checked empirically in section 4)

# Sensitivity analysis boundaries for section 7.4
RULE_VARIANTS = {"primary (2 s / 30 s)": (2.0, 30.0), "short (1 s / 20 s)": (1.0, 20.0),
                 "long (5 s / 40 s)": (5.0, 40.0)}

# ---- relaxation model -------------------------------------------------------------------
D_EXP = 0.5                # shear-aggregation exponent, fixed at 1/2 in all four papers
TAU_LAMBDA_LIT = (1.13, 0.24)   # s, mean and SD over 12 donors (Frontiers in Physics 2022, Table 6)
G_LIT_V1 = 0.29            # Pa, G of ethixo-EVP version 1, Donor 3 (Armstrong et al. 2021, Table 3) -- same model
G_LIT_R = (0.15, 0.05)     # Pa, rouleaux modulus G_R of the tensorial t-ESSTV, 12 donors (Frontiers 2022) -- related model
MODEL_MAX_NRMSE = 0.10     # a step is 'poorly fitted' if RMS error > 10 % of that step's maximum stress

## 1. Load the step-up/down blocks

Every workbook uses the ARES-G2 export layout: row 1 holds the test label (e.g. `5-->2.5 (1/s)`), row 2 the variable names (`Step time`, `Stress`, `Strain`, `Shear rate`, …) and row 3 the units, with one block of columns per test. The reader below is **non-destructive**: it never edits the workbook and keeps the workbook, sheet and block number for every row, so each measurement can be traced back to its source.

APR18 has its header one row lower than the other files; the reader detects this automatically.

In [ ]:
def storage_hours(sheet_title):
    """Storage time of the sample in hours, parsed from the sheet name ('t0', 't12', '24hr'); 0 if absent."""
    s = sheet_title.lower()
    m = re.search(r"\bt\s?(\d+)\b", s) or re.search(r"(\d+)\s?h", s)
    return float(m.group(1)) if m else 0.0

def parse_step_label(label):
    m = re.search(r"([\d.]+)\s*-->\s*([\d.]+)", label.replace(" ", ""))
    return (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)

def read_step_blocks(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    out = []
    for ws in wb.worksheets:
        title = ws.title.strip()
        if "step" not in title.lower() or any(k in title.lower() for k in EXCLUDE_SHEETS):
            continue
        R = list(ws.iter_rows(values_only=True))
        if len(R) < 4:
            continue
        W = max(len(r) for r in R[:4])
        cell = lambda r, j: R[r][j] if j < len(R[r]) else None
        hr = 1 if any(str(v).strip().lower().startswith("step time") for v in R[1] if v) else 2
        hdr = [str(cell(hr, j)).strip().lower() if cell(hr, j) is not None else "" for j in range(W)]
        data = R[hr + 2:]
        tcols = [j for j, h in enumerate(hdr) if h.startswith("step time")]
        for k, c in enumerate(tcols):
            nxt = tcols[k + 1] if k + 1 < len(tcols) else W
            label = next((str(cell(0, j)).strip() for j in range(c, nxt) if cell(0, j) not in (None, "")), "")
            cols = {}
            for j in range(c, nxt):
                h = hdr[j]
                key = ("time" if h.startswith("step time") else "stress" if "stress" in h else
                       "shear_rate" if "shear rate" in h else "strain" if h.startswith("strain") else
                       "viscosity" if "viscosity" in h else None)
                if key and key not in cols:
                    cols[key] = np.array([r[j] if j < len(r) and isinstance(r[j], (int, float)) else np.nan
                                          for r in data], float)
            if "stress" not in cols:
                continue
            df = pd.DataFrame(cols)
            df = df[np.isfinite(df.time) & (df.time > 0) & np.isfinite(df.stress)].reset_index(drop=True)
            if len(df) < 10:
                continue
            g0, g1 = parse_step_label(label)
            out.append(dict(workbook=path.name, sheet=title, block=k + 1, label=label,
                            storage_h=storage_hours(title), rate_from=g0, rate_to=g1, data=df))
    return out

blocks = []
for p in sorted(DATA_DIR.glob("*.xlsx")):
    if any(k in p.name for k in EXCLUDE_WORKBOOKS):
        continue
    blocks += read_step_blocks(p)

# Remove blocks whose whole stress trace repeats an earlier block (copy-pasted inside the workbooks; in one case,
# DEC17 24 h '.1-->1', only the stress column was copied and the shear-rate column differs)
seen = {}
for b in blocks:
    h = hashlib.md5(np.round(b["data"].stress.values, 9).tobytes()).hexdigest()
    b["duplicate_of"] = seen.get(h, "")
    seen.setdefault(h, f'{b["workbook"]}::{b["sheet"]}::{b["label"]}')
dups = [(b["workbook"], b["sheet"], b["label"], b["duplicate_of"]) for b in blocks if b["duplicate_of"]]
print(f"{len(dups)} blocks removed as stress-trace duplicates:")
for d_ in dups:
    print("  ", d_[0][:14], "|", d_[1], "|", d_[2], "  ==  ", d_[3].split("::", 1)[1])
blocks = [b for b in blocks if not b["duplicate_of"]]

for i, b in enumerate(blocks):
    b["step_id"] = f"S{i+1:03d}"
    m = re.search(r"(\d{1,2}[A-Z]{3}\d{0,2}|[A-Z]{3}\d{2})", b["workbook"])       # e.g. JUN18, 12FEB20, DEC17
    b["sample"] = m.group(1) if m else b["workbook"]
    b["n_points"] = len(b["data"])
    b["high_res"] = b["n_points"] > 1000

inv = pd.DataFrame([{k: v for k, v in b.items() if k != "data"} for b in blocks])
print(f"{len(inv)} step blocks from {inv.workbook.nunique()} workbooks "
      f"({inv.high_res.sum()} high-resolution, {(~inv.high_res).sum()} log-spaced)")
inv.groupby("sample").agg(blocks=("step_id", "size"), storage_times=("storage_h", "nunique"))

### 1.1 Steady-state flow curves

Each workbook also has a steady-state flow curve (16 shear rates from 0.1 to 700 s⁻¹). The papers fit the steady-state parameters to this curve first, then fit the transient parameter to the steps. The flow curve is also used here as an independent check of the pre-step baseline and of the final plateau. Where a workbook has several flow curves (DEC17 at 0/12/24 h, FEB18 at 0/6/24 h), the one matching the step sheet's storage time is used.

In [ ]:
def read_flow_curves(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    curves = []
    for ws in wb.worksheets:
        if not re.search(r"\bSS\b|steady", ws.title, re.I):
            continue
        R = list(ws.iter_rows(values_only=True))
        for r_i in range(3):
            for j, v in enumerate(R[r_i]):
                if v is None or "shear rate" not in str(v).lower():
                    continue
                above = " ".join(str(R[k][j]) for k in range(r_i) if j < len(R[k]) and R[k][j])
                xs, ys = [], []
                for r in R[r_i + 1:]:
                    x = r[j] if j < len(r) else None; y = r[j + 1] if j + 1 < len(r) else None
                    if not isinstance(x, (int, float)) or not isinstance(y, (int, float)) or (xs and x <= xs[-1]):
                        break
                    xs.append(x); ys.append(y)
                if len(xs) >= 8:
                    m = re.search(r"(\d+)", above)
                    curves.append(dict(storage_h=float(m.group(1)) if m else 0.0,
                                       rate=np.array(xs), stress=np.array(ys)))
    return curves

flow = {p.name: read_flow_curves(p) for p in sorted(DATA_DIR.glob("*.xlsx"))
        if not any(k in p.name for k in EXCLUDE_WORKBOOKS)}

def flow_curve_for(workbook, storage_h):
    cs = flow.get(workbook, [])
    match = [c for c in cs if c["storage_h"] == storage_h]
    return (match or cs or [None])[0]

def flow_stress(fc, rate):
    """Log-log interpolation of the flow-curve stress at a given shear rate."""
    if fc is None or not np.isfinite(rate) or rate <= 0:
        return np.nan
    return float(np.exp(np.interp(np.log(rate), np.log(fc["rate"]), np.log(np.clip(fc["stress"], 1e-6, None)))))

print({k: [c["storage_h"] for c in v] for k, v in flow.items()})

## 2. What a step block looks like

Before building features, look at the raw signal. Two things matter for everything that follows:

1. **The first few milliseconds are still at the starting rate.** The motor changes speed about 5–20 ms into the block, so the first 3–6 points give a genuine pre-step baseline.
2. **The shear rate rings.** The rate briefly overshoots the target (for example it reaches −0.16 s⁻¹ on a 5 → 0.1 step) before settling at about 0.05 s. Stress in that window reflects motor inertia, not the blood (the same instrument-inertia issue discussed by Das et al. 2024, cited in the Semester 1 report), so it is excluded from all features.

In [ ]:
ex = next(b for b in blocks if b["sample"] == "JUL18" and b["label"].startswith("5-->.1"))
d = ex["data"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].semilogx(d.time, d.shear_rate, "o-", ms=3); ax[0].axhline(ex["rate_to"], color="k", lw=0.8, ls="--")
ax[0].set(xlabel="time since block start (s)", ylabel="shear rate (1/s)", title=f'{ex["sample"]} {ex["label"]}: applied rate')
ax[1].semilogx(d.time, d.stress, "o-", ms=3, color="C3")
ax[1].set(xlabel="time since block start (s)", ylabel="stress (Pa)", title="measured stress")
for a in ax: a.axvspan(1e-3, MIN_INSTRUMENT_WINDOW, color="grey", alpha=0.15)
fig.tight_layout(); fig.savefig(FIG / "01_raw_step_block.png", dpi=200); plt.show()

## 3. Verify every step event against the protocol

For each block:

* **Measured initial rate** = median shear rate of the points before the motor starts to move.
* **Step time** t_step = first time the rate has moved more than half-way from the initial to the final value.
* **Measured final rate** = median shear rate for t ≥ 1 s.
* **Rate settling time** = time after which the rate stays within 5 % of the target (the end of the instrument window).
* **Rate overshoot** = largest excursion of the rate beyond the target, as a fraction of the step size.
* **Magnitude** is reported as the rate ratio r = final/initial and as log₁₀ r; **direction** is `up` if r > 1.

A block passes if both measured rates are within 5 % of its label. High-resolution blocks start after the step, so their initial rate cannot be measured and is taken from the label (flagged).

In [ ]:
def verify_step(b):
    d = b["data"]; t = d.time.values
    g0_lab, g1_lab = b["rate_from"], b["rate_to"]
    res = dict(step_id=b["step_id"])
    if "shear_rate" not in d or d.shear_rate.isna().all():
        # no rate channel: time is zeroed at the step, so t_step = 0 and the instrument window is the minimum
        res.update(rate_from_meas=np.nan, rate_to_meas=np.nan, has_pre_step=False, t_step=0.0,
                   t_rate_settle=MIN_INSTRUMENT_WINDOW, rate_overshoot_frac=np.nan,
                   qc_rate="label only (no shear-rate channel)")
        return res
    g = d.shear_rate.values
    g1 = np.nanmedian(g[t >= 1.0])
    has_pre = t[0] < 0.004 and abs(g[0] - g1) > 0.3 * abs(g1_lab - g0_lab)
    g0 = np.nanmedian(g[:3]) if has_pre else g0_lab
    half = g0 + 0.5 * (g1 - g0)
    moved = np.where(np.sign(g1 - g0) * (g - half) > 0)[0]
    t_step = t[moved[0]] if has_pre and len(moved) else 0.005     # median observed value when not measurable
    tol = np.maximum(RATE_SETTLE_TOL * abs(g1), 0.01)
    bad = np.where(np.abs(g - g1) > tol)[0]
    t_rs = t[bad[-1] + 1] if len(bad) and bad[-1] + 1 < len(t) else t[0]
    after = g[t > t_step]
    excess = (np.max(after) - g1) if g1 > g0 else (g1 - np.min(after))
    ok0 = has_pre and abs(g0 / g0_lab - 1) <= RATE_TOL
    ok1 = abs(g1 / g1_lab - 1) <= RATE_TOL
    res.update(rate_from_meas=g0, rate_to_meas=g1, has_pre_step=has_pre, t_step=t_step,
               t_rate_settle=max(t_rs, MIN_INSTRUMENT_WINDOW), rate_overshoot_frac=excess / abs(g1 - g0),
               qc_rate=("ok" if ok0 and ok1 else
                        "final ok, initial from label (no pre-step samples)" if ok1 and not has_pre else
                        "MISMATCH with label"))
    return res

ev = inv.merge(pd.DataFrame([verify_step(b) for b in blocks]), on="step_id")
ev["ratio"] = ev.rate_to / ev.rate_from
ev["log10_ratio"] = np.log10(ev.ratio)
ev["direction"] = np.where(ev.ratio > 1, "up", "down")
ev["magnitude_abs"] = (ev.rate_to - ev.rate_from).abs()

print(ev.qc_rate.value_counts().to_string())
cols = ["step_id", "sample", "sheet", "label", "storage_h", "rate_from", "rate_to", "rate_from_meas", "rate_to_meas",
        "direction", "log10_ratio", "t_step", "t_rate_settle", "rate_overshoot_frac", "qc_rate"]
ev[cols].to_csv(OUT / "step_events.csv", index=False)
ev[cols].head(12)

In [ ]:
bad = ev[ev.qc_rate.str.startswith("MISMATCH")]
print(f"{len(bad)} blocks disagree with their label:")
bad[cols]

**Result.** 133 of 136 blocks match their protocol label. The three mismatches are kept out of all later sections:
* **APR18 `5-->0.1`**: the measured final rate is 0.50 s⁻¹, so this block is really a 5 → 0.5 step. It looks like a labelling error in the workbook and should be reported to the client.
* **14MAR19 `.1-->10`**: the rate before the step is −0.30 s⁻¹, not 0.1, so the sample was not at the stated starting state (the workbook also contains a flow-reversal test, which may have preceded it; this cannot be confirmed from the file).
* **12FEB20 `.1-->20`**: the initial rate is 0.105 s⁻¹, 5.1 % off, only just outside the tolerance. It is excluded for consistency with the fixed rule.

**Instrument window.** The table below summarises how long the motor takes to settle. Stress before `t_rate_settle` is excluded in every later section.

In [ ]:
ev.groupby("direction")[["t_step", "t_rate_settle", "rate_overshoot_frac"]].describe(percentiles=[.5]).round(3)

## 4. Fixed transient / stable rule

Defined in the configuration cell and justified by the literature time scale τ_λ ≈ 1.1 s:

* **Transient window W_T** = [rate settling time, 2 s]
* **Stable window W_S** = [30 s, end of test (≈ 63 s)]
* **Stable value** σ_stable = median stress in W_S
* **Defensible stable state**: the stress drift across W_S (from a linear fit of σ against log t) is less than 10 % of σ_stable (or less than the noise floor). Measurements that fail this **never reach a defensible stable state** and are flagged, not dropped.

The rule depends only on time, not on the stress feature being tested, so §7.3 is not circular.

### 4.1 Noise floor check
Before applying the rule, check that the fixed absolute noise floor (1 mPa) is realistic, using the scatter of stress about a smooth trend inside W_S.

In [ ]:
def window(d, t0, t1):
    return d[(d.time >= t0) & (d.time <= t1)]

noise = []
for b in blocks:
    w = window(b["data"], STABLE_START, np.inf)
    if len(w) >= 4:
        x = np.log10(w.time.values); y = w.stress.values
        resid = y - np.polyval(np.polyfit(x, y, 1), x)
        noise.append(dict(step_id=b["step_id"], high_res=b["high_res"], resid_sd=np.std(resid, ddof=2),
                          stable_level=np.median(y)))
noise = pd.DataFrame(noise)
print(noise.groupby("high_res").resid_sd.describe(percentiles=[.5, .9]).round(5).to_string())
print(f"\nConfigured NOISE_PA = {NOISE_PA} Pa")

The log-spaced (65-point) blocks are much smoother than the high-resolution ones because the instrument averages over each logarithmic interval. The high-resolution 5 → 0.25 s⁻¹ blocks are dominated by noise at this low stress (≈ 0.01 Pa). **This matters for the Haemograph feature set:** at low shear rates, raw fast sampling needs averaging before transient features are stable.

To keep all measurements comparable (the framework's *same windows, same preprocessing* rule), high-resolution blocks are **averaged into the same logarithmic time bins** as the 65-point blocks before any feature is computed. The raw high-resolution series stays available to Member 4.

In [ ]:
ref_grid = next(b for b in blocks if not b["high_res"])["data"].time.values
edges = np.sqrt(ref_grid[1:] * ref_grid[:-1]); edges = np.r_[ref_grid[0] / 1.2, edges, ref_grid[-1] * 1.2]

def log_bin(d):
    idx = np.digitize(d.time.values, edges) - 1
    g = d.assign(bin=idx).groupby("bin").mean(numeric_only=True)
    return g[g.index.isin(range(len(ref_grid)))].reset_index(drop=True)

for b in blocks:
    b["an"] = log_bin(b["data"]) if b["high_res"] else b["data"]   # 'analysis' series
print("high-res blocks after binning:", [len(b["an"]) for b in blocks if b["high_res"]])

In [ ]:
def windows_for(b, e, t_trans_end=TRANSIENT_END, t_stable_start=STABLE_START):
    d = b["an"]
    wt = window(d, e.t_rate_settle, t_trans_end)
    ws = window(d, t_stable_start, np.inf)
    s_stable = np.median(ws.stress) if len(ws) else np.nan
    if len(ws) >= 3:
        x = np.log10(ws.time.values)
        slope = np.polyfit(x, ws.stress.values, 1)[0]
        drift = slope * (x[-1] - x[0])
    else:
        drift = np.nan
    drift_frac = abs(drift) / max(abs(s_stable), NOISE_PA) if np.isfinite(drift) else np.nan
    return dict(n_transient=len(wt), n_stable=len(ws), sigma_stable=s_stable, stable_drift_Pa=drift,
                stable_drift_frac=drift_frac,
                stable_ok=bool(np.isfinite(drift_frac) and (drift_frac <= STABLE_MAX_DRIFT or abs(drift) <= NOISE_PA)),
                below_noise=bool(abs(s_stable) < 3 * NOISE_PA))

evi = ev.set_index("step_id")
win = pd.DataFrame([dict(step_id=b["step_id"], **windows_for(b, evi.loc[b["step_id"]])) for b in blocks])
ev = ev.drop(columns=[c for c in win.columns if c != "step_id" and c in ev], errors="ignore").merge(win, on="step_id")
ev[["step_id", "sample", "label", "n_transient", "n_stable", "sigma_stable", "stable_drift_frac",
    "stable_ok", "below_noise"]].to_csv(OUT / "step_windows.csv", index=False)

print(f"Defensible stable state reached: {ev.stable_ok.sum()} / {len(ev)} measurements")
print(f"Stable stress within 3x noise floor: {ev.below_noise.sum()}")
ev.assign(final_rate=ev.rate_to).groupby(["direction", "final_rate"]).stable_ok.agg(["mean", "size"]).round(2).T

**Result.** 118 of 136 measurements (87 %) reach a defensible stable state. The table shows **which step conditions fail to settle within 63 s**: only 57 % of steps down to 0.1 s⁻¹ settle, against 83–100 % for every other final rate. Steps *down* to low shear rates (0.1–0.25 s⁻¹) are the usual failures: stress is still rising at the end of the test, which is consistent with continuing rouleaux aggregation (Pincot et al. 2023: aggregation is much slower than breakdown). Their “final” value is therefore a lower bound, and features that divide by it are marked as less reliable in §8.

## 5. Conventional transient features

All features are computed on t ≥ rate settling time (instrument window excluded). σ_stable is the W_S median from §4.

| Feature | Definition | Units |
|---|---|---|
| `sigma_pre` | median stress before the step (pre-step baseline) | Pa |
| `sigma_pre_fc` | flow-curve stress at the initial rate (independent baseline check) | Pa |
| `sigma_imm` | first stress value after the instrument window (immediate response) | Pa |
| `sigma_peak`, `t_peak` | maximum stress after the window and its time (time to peak) | Pa, s |
| `sigma_min`, `t_min` | minimum stress after the window and its time | Pa, s |
| `overshoot` | (σ_peak − σ_stable)/σ_stable | – |
| `undershoot` | (σ_stable − σ_min)/σ_stable | – |
| `jump_signed` | (σ_imm − σ_stable)/σ_stable, **signed relaxation amplitude** | – |
| `relax_amp` | σ_imm − σ_stable | Pa |
| `init_slope` | slope of σ/σ_stable against log₁₀ t between the window and 1 s | per decade |
| `settling_time` | time after which σ stays within ±10 % of σ_stable (NaN if never) | s |
| `area_log` | ∫ (σ − σ_stable)/σ_stable d(ln t) from the window to the end: net excess (+) or deficit (−) of stress relative to steady state | – |
| `plateau_vs_fc` | σ_stable / flow-curve stress at the final rate (checks the plateau against steady-shear data) | – |
| `visc_ratio` | (σ_stable/γ̇₁)/(σ_pre/γ̇₀), apparent viscosity after/before | – |

In [ ]:
def features(b, e):
    d = b["an"]; t = d.time.values; s = d.stress.values
    fc = flow_curve_for(b["workbook"], b["storage_h"])
    ss = e.sigma_stable
    m = t >= e.t_rate_settle
    tm, sm = t[m], s[m]
    pre = s[t < 0.5 * e.t_step] if e.has_pre_step else np.array([])
    f = dict(step_id=b["step_id"])
    f["sigma_pre"] = np.median(pre) if len(pre) else np.nan
    f["sigma_pre_fc"] = flow_stress(fc, e.rate_from)
    f["sigma_final_fc"] = flow_stress(fc, e.rate_to)
    f["sigma_imm"] = sm[0]
    i_pk, i_mn = np.argmax(sm), np.argmin(sm)
    f.update(sigma_peak=sm[i_pk], t_peak=tm[i_pk], sigma_min=sm[i_mn], t_min=tm[i_mn])
    denom = abs(ss) if abs(ss) > NOISE_PA else np.nan
    f["overshoot"] = (sm[i_pk] - ss) / denom
    f["undershoot"] = (ss - sm[i_mn]) / denom
    f["jump_signed"] = (sm[0] - ss) / denom
    f["relax_amp"] = sm[0] - ss
    early = (tm <= 1.0)
    f["init_slope"] = (np.polyfit(np.log10(tm[early]), sm[early] / denom, 1)[0]
                       if early.sum() >= 3 and np.isfinite(denom) else np.nan)
    outside = np.where(np.abs(sm - ss) > max(SETTLE_BAND * abs(ss), NOISE_PA))[0]
    if len(outside) == 0:
        f["settling_time"] = tm[0]
    elif outside[-1] + 1 < len(tm) and tm[outside[-1] + 1] < STABLE_START:
        f["settling_time"] = tm[outside[-1] + 1]
    else:
        f["settling_time"] = np.nan
    f["area_log"] = trapezoid((sm - ss) / denom, np.log(tm))
    f["plateau_vs_fc"] = ss / f["sigma_final_fc"]
    f["baseline_vs_fc"] = f["sigma_pre"] / f["sigma_pre_fc"]
    f["visc_ratio"] = ((ss / e.rate_to) / (f["sigma_pre"] / e.rate_from)
                       if len(pre) and abs(f["sigma_pre"]) >= 3 * NOISE_PA else np.nan)   # baseline must exceed noise
    return f

evs = ev.set_index("step_id")
feat = pd.DataFrame([features(b, evs.loc[b["step_id"]]) for b in blocks])
steps = ev.merge(feat, on="step_id")
FEATURES = ["jump_signed", "overshoot", "undershoot", "init_slope", "settling_time", "t_peak", "t_min",
            "area_log", "relax_amp", "visc_ratio"]
steps.to_csv(OUT / "step_features.csv", index=False)
steps[["step_id", "sample", "label", "direction", "sigma_pre", "sigma_imm", "sigma_stable"] + FEATURES].head(10).round(4)

### 5.1 Validate the baseline and plateau against steady-shear data
If the pre-step stress and the final plateau are genuine steady states, they should agree with the independent flow curve measured on the same sample. Ratios near 1 support the feature definitions; large departures identify blocks whose baseline or plateau cannot be trusted.

In [ ]:
chk = steps[["baseline_vs_fc", "plateau_vs_fc"]].replace([np.inf, -np.inf], np.nan)
print(chk.describe(percentiles=[.1, .5, .9]).round(2).to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for a, (col, ref, ttl) in zip(ax, [("sigma_pre", "sigma_pre_fc", "pre-step baseline"),
                                   ("sigma_stable", "sigma_final_fc", "final plateau (W_S median)")]):
    for dirn, mk in [("up", "^"), ("down", "v")]:
        q = steps[steps.direction == dirn]
        a.loglog(q[ref], q[col].clip(lower=1e-4), mk, ms=5, alpha=0.7, label=f"step {dirn}")
    lim = [1e-3, 3]; a.plot(lim, lim, "k--", lw=0.8)
    a.set(xlabel="flow-curve stress at same rate (Pa)", ylabel="step-test stress (Pa)", title=ttl); a.legend()
fig.tight_layout(); fig.savefig(FIG / "02_baseline_plateau_vs_flowcurve.png", dpi=200); plt.show()

**Result.** The pre-step baseline agrees with the flow curve (median ratio 1.07). The final plateau is systematically **about 13 % above** the flow curve (median 1.13, 90th percentile 2.0), with the largest departures at low shear rates. The step tests and the flow curve were run at different times on the same loaded sample, so this gap reflects time-dependence (and possibly sedimentation) of the blood. Features in this notebook are therefore normalised by each step's **own** plateau σ_stable, never by the flow curve.

## 6. Relaxation models (only where justified)

Two levels of model are fitted, both on t ≥ rate settling time.

### 6.1 Empirical single exponential
σ(t) = σ∞ + A·exp(−(t − t_step)/τ). This is a descriptive check on the time scale. A fit counts as **failed / not interpretable** if it does not converge, τ sits on a bound (0.01 s or 200 s), the 95 % confidence interval of τ is wider than τ itself, or R² < 0.5. Non-monotonic responses (for example 300 → 5 s⁻¹, where stress dips, rises and then decays) are expected to fail and are logged.

### 6.2 Thixo-elasto-visco-plastic model from the papers (ethixo-EVP, version 1)
From Armstrong et al. (2021), with n = 1 and the aggregation exponent d = ½ fixed as in all four papers.

Steady state (Eqs. 10 and 16):

$$\sigma_{ss}=\lambda_{ss}\,\sigma_{y0}+\left(\lambda_{ss}\,\eta_{ST}+\eta_\infty\right)\dot\gamma,\qquad
\lambda_{ss}=\frac{t_{r2}|\dot\gamma|^{1/2}+1}{t_{r1}|\dot\gamma|+t_{r2}|\dot\gamma|^{1/2}+1}$$

Transient (Eq. 11 for stress, Eq. 13 for structure, both driven by the plastic rate γ̇ₚ):

$$\frac{1}{G}\dot\sigma=\dot\gamma-\dot\gamma_p,\qquad
\dot\gamma_p=\frac{\max(0,\,|\sigma|-\lambda\sigma_{y0})}{\lambda\eta_{ST}+\eta_\infty}\,\mathrm{sign}(\sigma)$$

$$\dot\lambda=k_\lambda\left(-t_{r1}|\dot\gamma_p|\lambda+t_{r2}|\dot\gamma_p|^{1/2}(1-\lambda)+(1-\lambda)\right)$$

The elastic modulus G is kept because the data need it: after a step down, stress first *falls* for about 1 s (elastic relaxation) before rising again as structure rebuilds. A model without G cannot produce that dip. After a step to rate γ̇₁ the structure relaxes with an effective time τ_eff ≈ 1/(k_λ·(t_{r1}γ̇₁ + t_{r2}γ̇₁^{1/2} + 1)): **faster at high final rates (breakdown) and slower at low final rates (aggregation)**, which is the asymmetry the papers report.

**Fitting procedure (as in the papers):**
1. Initial guess: fit σy0, ηST, η∞, tr1, tr2 to the flow curve with the relative cost (Eq. 17).
2. Fit all seven parameters **simultaneously** to the flow curve and every step-up and step-down of that workbook, as done for the tensorial models in the Frontiers (2022) and Sci. Rep. (2021) papers. The cost is the least-squares form of Frontiers Eq. 12, with each step normalised by its own maximum stress. The ODEs are integrated with LSODA (the papers use MATLAB's ode23s; both are stiff solvers). Two starting points are used: the stage-1 result and the papers' typical values.
3. Goodness of fit per step is the NRMSE (RMS error / maximum stress of that step). A step is **poorly fitted** if NRMSE > 10 %, or > 2× noise/maximum stress for very low-stress steps, which cannot be fitted better than the noise.
4. Compare with the literature: τ_λ = 1.13 ± 0.24 s (Frontiers 2022, 12 donors) and 0.55–0.96 s (Sci. Rep. 2021, Donor 1); G = 0.29 Pa for this same model (JNNFM 2021, Donor 3). The Frontiers G_R = 0.15 ± 0.05 Pa belongs to the tensorial t-ESSTV model and is shown only as a related reference.

This takes roughly 10–20 s per workbook.

In [ ]:
def exp_model(t, s_inf, A, tau):
    return s_inf + A * np.exp(-t / tau)

def fit_exponential(b, e):
    d = b["an"]; m = d.time >= e.t_rate_settle
    t = d.time.values[m] - e.t_step; y = d.stress.values[m]
    rec = dict(step_id=b["step_id"], model="exponential")
    try:
        p0 = [np.median(y[-5:]), y[0] - np.median(y[-5:]), 1.0]
        p, cov = curve_fit(exp_model, t, y, p0=p0, bounds=([-np.inf, -np.inf, 0.01], [np.inf, np.inf, 200]), maxfev=20000)
        yhat = exp_model(t, *p); r2 = 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)
        se = np.sqrt(np.diag(cov))[2] if np.all(np.isfinite(cov)) else np.inf
        reasons = []
        if p[2] <= 0.0101 or p[2] >= 199: reasons.append("tau at bound")
        if not np.isfinite(se) or 1.96 * se > p[2]: reasons.append("tau CI wider than tau")
        if r2 < 0.5: reasons.append("R2 < 0.5")
        rec.update(s_inf=p[0], amp=p[1], tau=p[2], tau_se=se, r2=r2, rmse=np.sqrt(np.mean((y - yhat) ** 2)),
                   status="ok" if not reasons else "not interpretable", reason="; ".join(reasons))
    except Exception as err:
        rec.update(status="failed", reason=type(err).__name__)
    return rec

evs = ev.set_index("step_id")
fits_exp = pd.DataFrame([fit_exponential(b, evs.loc[b["step_id"]]) for b in blocks])
print(fits_exp.status.value_counts().to_string())
fits_exp.merge(ev[["step_id", "direction", "rate_from", "rate_to"]]).groupby(["direction", "rate_to"]).status.agg(
    lambda s: f"{(s == 'ok').sum()}/{len(s)} ok").unstack(0).fillna("")

In [ ]:
from scipy.integrate import solve_ivp

def lam_ss(g, tr1, tr2):                                   # Eq. 16
    g = np.abs(g); return (tr2 * g ** D_EXP + 1) / (tr1 * g + tr2 * g ** D_EXP + 1)

def sigma_ss(g, p):                                        # Eq. 10
    sy, eta_st, eta_inf, tr1, tr2 = p[:5]
    lam = lam_ss(g, tr1, tr2); return lam * sy + (lam * eta_st + eta_inf) * g

def simulate_step(t_eval, g0, g1, t_step, p):
    """ethixo-EVP version 1: Eq. 11 (n = 1) for stress and Eq. 13 for structure, driven by the plastic rate.
    Starts from steady state at the initial rate g0; the rate switches to g1 at t_step."""
    sy, eta_st, eta_inf, tr1, tr2, G, k_lam = p
    def rhs(t, z):
        s, lam = z
        g = g1 if t >= t_step else g0
        gp = max(0.0, abs(s) - lam * sy) / (lam * eta_st + eta_inf) * np.sign(s)     # plastic shear rate
        agp = abs(gp)
        return [G * (g - gp), k_lam * (-tr1 * agp * lam + tr2 * agp ** D_EXP * (1 - lam) + (1 - lam))]
    z0 = [sigma_ss(g0, p), lam_ss(g0, tr1, tr2)]
    sol = solve_ivp(rhs, (0, t_eval[-1]), z0, t_eval=t_eval, method="LSODA", rtol=1e-6, atol=[1e-8, 1e-8],
                    first_step=1e-5, max_step=t_eval[-1] / 20)
    if not sol.success or sol.y.shape[1] != len(t_eval):
        return np.full(len(t_eval), np.nan), np.full(len(t_eval), np.nan)
    return sol.y[0], sol.y[1]

def step_series(b, e):
    """Stress after the instrument window, with time measured from the step."""
    d = b["an"]; m = d.time >= e.t_rate_settle
    return d.time.values[m] - e.t_step, d.stress.values[m]

#                  sigma_y0  eta_ST  eta_inf  tr1   tr2    G     k_lambda
LOW = np.log([1e-6, 1e-4, 1e-4, 1e-3, 1e-4, 1e-3, 1e-3])
HIGH = np.log([1.0, 1.0, 0.05, 1e3, 1e3, 1e2, 1e2])
P_LIT = np.array([0.003, 0.02, 0.003, 0.5, 0.2, 0.3, 1.0])   # start near the papers' reported values

def fit_steady(x, y):
    """Stage 1 (initial guess, as in the papers): Eq. 17 relative cost on the flow curve only."""
    r = least_squares(lambda q: (y - sigma_ss(x, np.exp(q))) / y, np.log(P_LIT[:5]), bounds=(LOW[:5], HIGH[:5]))
    return np.exp(r.x)

def joint_residuals(q, x, y, series):
    """Stage 2: least-squares form of Frontiers Eq. 12 -- flow-curve residuals normalised point-wise, each step's
    residuals by that step's maximum stress, and every experiment weighted equally."""
    p = np.exp(q)
    r = [(y - sigma_ss(x, p)) / y / np.sqrt(len(y))]
    for t, s, g0, g1, ts in series:
        f, _ = simulate_step(t, g0, g1, ts, p)
        rr = (s - f) / np.max(np.abs(s)) / np.sqrt(len(s) * len(series))
        r.append(np.where(np.isfinite(rr), rr, 10.0))           # solver failure -> large penalty
    return np.concatenate(r)

model_rows, model_curves = [], {}
for (wb, sh), grp in ev[ev.qc_rate != "MISMATCH with label"].groupby(["workbook", "storage_h"]):
    fc = flow_curve_for(wb, sh)
    if fc is None: continue
    x, y = fc["rate"][fc["stress"] > 0], fc["stress"][fc["stress"] > 0]
    ser = []
    for sid in grp.step_id:
        e = evs.loc[sid]; d = next(b for b in blocks if b["step_id"] == sid)["an"]
        m = d.time >= e.t_rate_settle
        ser.append((d.time.values[m], d.stress.values[m], e.rate_from, e.rate_to, e.t_step))
    best = None
    for st in (np.r_[fit_steady(x, y), P_LIT[5:]], P_LIT):
        r = least_squares(joint_residuals, np.log(st), args=(x, y, ser), bounds=(LOW, HIGH), max_nfev=300)
        if best is None or r.cost < best.cost: best = r
    p = np.exp(best.x)
    # conditional profile of k_lambda (other parameters held at the optimum): k values within 10 % of the minimum cost
    grid = np.exp(np.linspace(LOW[6], HIGH[6], 41))
    prof = np.array([np.sum(joint_residuals(np.log(np.r_[p[:6], kk]), x, y, ser) ** 2) for kk in grid])
    within = grid[prof <= prof.min() * 1.10]
    per = []
    for sid, (t, s, g0, g1, ts) in zip(grp.step_id, ser):
        f, lam = simulate_step(t, g0, g1, ts, p)
        nrmse = np.sqrt(np.mean(((s - f) / np.max(np.abs(s))) ** 2))
        per.append(dict(step_id=sid, nrmse_model=nrmse,
                        fit_threshold=max(MODEL_MAX_NRMSE, 2 * NOISE_PA / np.max(np.abs(s))),
                        tau_eff=1 / (p[6] * (p[3] * g1 + p[4] * g1 ** D_EXP + 1))))
        model_curves[sid] = (t - ts, f, lam)
    per = pd.DataFrame(per).set_index("step_id")
    sel = steps.step_id.isin(per.index)
    for c in per.columns:
        steps.loc[sel, c] = steps.loc[sel, "step_id"].map(per[c]).values
    rss = np.sum(best.fun ** 2); n = len(best.fun); k_par = 7
    model_rows.append(dict(workbook=wb, sample=grp["sample"].iloc[0], storage_h=sh, n_steps=len(ser),
                           sigma_y0=p[0], eta_ST=p[1], eta_inf=p[2], tr1=p[3], tr2=p[4], G=p[5], k_lambda=p[6],
                           tau_lambda=1 / p[6], tau_lambda_lo=1 / within.max(), tau_lambda_hi=1 / within.min(),
                           Fcost_ss=np.sqrt(np.mean(((y - sigma_ss(x, p)) / y) ** 2)),
                           median_step_nrmse=per.nrmse_model.median(),
                           steps_fitted_ok=f"{(per.nrmse_model <= per.fit_threshold).sum()}/{len(per)}",
                           AIC=n * np.log(rss / n) + 2 * k_par, BIC=n * np.log(rss / n) + k_par * np.log(n),
                           k_at_bound=bool(best.x[6] <= LOW[6] + 0.05 or best.x[6] >= HIGH[6] - 0.05)))

models = pd.DataFrame(model_rows)
models.to_csv(OUT / "relaxation_fits.csv", index=False)
models.drop(columns="workbook").round(4)

In [ ]:
fails = pd.concat([
    fits_exp[fits_exp.status != "ok"].merge(ev[["step_id", "sample", "label"]])[["step_id", "sample", "label", "model", "status", "reason"]],
    steps[steps.nrmse_model > steps.fit_threshold].assign(model="ethixo-EVP", status="poor fit",
                                       reason=lambda x: "NRMSE = " + x.nrmse_model.round(3).astype(str))[
        ["step_id", "sample", "label", "model", "status", "reason"]],
    models[models.k_at_bound].assign(step_id="(all)", label="(workbook)", model="ethixo-EVP",
                                     status="k_lambda at bound", reason="not identifiable")[
        ["step_id", "sample", "label", "model", "status", "reason"]]])
fails.to_csv(OUT / "fit_failure_log.csv", index=False)
print(f"{len(fails)} entries in the failure log")
fails.groupby(["model", "status"]).size()

### 6.3 Validation against the literature parameters

In [ ]:
mu, sd = TAU_LAMBDA_LIT
okm = models[~models.k_at_bound]
print(f"tau_lambda (this data): median {okm.tau_lambda.median():.2f} s, IQR {okm.tau_lambda.quantile(.25):.2f}-"
      f"{okm.tau_lambda.quantile(.75):.2f} s  (n = {len(okm)} workbook x storage fits, {okm['sample'].nunique()} workbooks)")
print(f"   literature: {mu} +- {sd} s (Frontiers 2022, 12 donors); 0.55-0.96 s (Sci. Rep. 2021, Donor 1)")
print(f"   fits within literature mean +- 2 SD: {((okm.tau_lambda - mu).abs() <= 2 * sd).sum()} / {len(okm)}")
print(f"G (this data): median {okm.G.median():.3f} Pa, range {okm.G.min():.3f}-{okm.G.max():.3f} Pa"
      f";  literature: G = {G_LIT_V1} Pa (same model, JNNFM 2021), G_R = {G_LIT_R[0]} +- {G_LIT_R[1]} Pa (t-ESSTV, Frontiers 2022)")
print(f"steps fitted within threshold: {(steps.nrmse_model <= steps.fit_threshold).sum()} / {steps.nrmse_model.notna().sum()}")
print("\nfit quality by final shear rate (median NRMSE):")
print(steps.groupby(["direction", "rate_to"]).nrmse_model.median().unstack(0).round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
yy = np.arange(len(models)); names = [f'{s_} t={int(h)}h' for s_, h in zip(models["sample"], models.storage_h)]
ax[0].errorbar(models.tau_lambda, yy, xerr=[models.tau_lambda - models.tau_lambda_lo, models.tau_lambda_hi - models.tau_lambda],
               fmt="o", capsize=3)
ax[0].axvspan(mu - 2 * sd, mu + 2 * sd, color="C2", alpha=0.15, label="literature mean ± 2 SD")
ax[0].set(xscale="log", xlabel="τ_λ = 1/k_λ (s)  (bars: conditional profile interval)"); ax[0].legend(loc="lower left", fontsize=7)
ax[1].plot(models.G, yy, "s", color="C3")
ax[1].axvspan(G_LIT_R[0] - 2 * G_LIT_R[1], G_LIT_R[0] + 2 * G_LIT_R[1], color="C2", alpha=0.15, label="G_R, t-ESSTV (Frontiers) ± 2 SD")
ax[1].axvline(G_LIT_V1, color="k", ls="--", lw=1, label="G, same model (JNNFM 2021)")
ax[1].set(xlabel="G (Pa)"); ax[1].legend(loc="lower right", fontsize=7)
ax[0].set_yticks(yy, names)
fig.tight_layout(); fig.savefig(FIG / "03_tau_lambda_vs_literature.png", dpi=200); plt.show()

### 6.4 Representative fits (papers' figure layout)
Step-downs from 5 s⁻¹ and step-ups from 0.25 s⁻¹, with the model's structure parameter λ(t) underneath, as in Armstrong et al. (2021) Figs. 3–4 and Frontiers (2022) Fig. 5. The shaded bands are the transient (orange) and stable (green) windows.

In [ ]:
def plot_family(sample, storage_h=0.0, start_down=5.0, start_up=0.25, fname="04_representative_fits.png"):
    q = steps[(steps["sample"] == sample) & (steps.storage_h == storage_h)]
    fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))
    for col, (dirn, g0) in enumerate([("down", start_down), ("up", start_up)]):
        sub = q[(q.direction == dirn) & np.isclose(q.rate_from, g0)].sort_values("rate_to")
        for i, (_, r) in enumerate(sub.iterrows()):
            b = next(bb for bb in blocks if bb["step_id"] == r.step_id)
            t, y = step_series(b, r)
            ax[0, col].semilogx(t, y, "o", ms=3, color=f"C{i}", label=f"{g0:g} → {r.rate_to:g} 1/s")
            if r.step_id in model_curves:
                tm, fm, lam = model_curves[r.step_id]
                ax[0, col].semilogx(tm, fm, "-", color=f"C{i}", lw=1.2)
                ax[1, col].semilogx(tm, lam, "-", color=f"C{i}")
        ax[0, col].axvspan(0.05, TRANSIENT_END, color="C1", alpha=0.08); ax[0, col].axvspan(STABLE_START, 70, color="C2", alpha=0.08)
        ax[0, col].set(title=f"{sample}: steps {dirn} from {g0:g} 1/s (points = data, lines = model)", ylabel="stress (Pa)")
        ax[0, col].legend(fontsize=7); ax[1, col].set(xlabel="time since step (s)", ylabel="structure λ")
    fig.tight_layout(); fig.savefig(FIG / fname, dpi=200); plt.show()

plot_family("JUN18")

**Result (relaxation models).**
* **Single exponential:** only 49 % of steps give an interpretable τ, so it is classed **unusable** as a transient metric. Step-down responses are non-monotonic (an elastic dip, then structural recovery), which one exponential cannot describe.
* **ethixo-EVP:** converges for all 13 workbook × storage-time sets with no parameter on a bound. **τ_λ has a median of 0.64 s (IQR 0.27–1.16 s)**, the same order as the literature (1.13 ± 0.24 s; 0.55–0.96 s), with 5 of 13 fits inside the literature mean ± 2 SD. **G has a median of 0.27 Pa (range 0.19–0.47 Pa)**, close to the 0.29 Pa the JNNFM paper reports for the same model. This independently reproduces the papers' parameter scale on the client data.
* **Where it fails:** fits are good at high final rates (median NRMSE 0.05–0.08 for final rates ≥ 10 s⁻¹) and poor at low final rates (0.17–0.21 at 0.1–0.25 s⁻¹). The figure shows why: after a step down, stress keeps drifting for 10–60 s, a process slower than the model's single structural time scale. This is consistent with the multi-time-scale thixotropy the JNNFM paper discusses (Wei et al., ML-IKH), and a single-λ model cannot capture it. Only 50 of 133 steps (38 %) are fitted within the pre-set threshold, so **per-step model outputs are not interpreted**; only the per-workbook τ_λ and G are used.

## 7. Do the features distinguish magnitude, direction and phase?

**Independent unit = workbook.** Each question is answered in two stages: a statistic within each workbook, then a test across workbooks (sign test or Wilcoxon signed-rank). With about 10 workbooks, all results are **exploratory** and are reported with effect sizes and counts, not just p-values.

Measurements flagged `MISMATCH with label` are excluded. Measurements without a defensible stable state are kept but marked, because features normalised by σ_stable are less reliable for them.

In [ ]:
A = steps[steps.qc_rate != "MISMATCH with label"].copy()
A["unit"] = A["sample"] + "_t" + A.storage_h.astype(int).astype(str)     # workbook x storage time
print(f"analysis set: {len(A)} steps, {A['sample'].nunique()} workbooks (independent units), {A.unit.nunique()} workbook x storage units")

def sign_test(x):
    x = np.asarray(x)[np.isfinite(x) & (np.asarray(x) != 0)]
    if len(x) == 0: return np.nan, 0, 0
    k = (x > 0).sum(); return stats.binomtest(k, len(x), 0.5).pvalue, k, len(x)

### 7.1 Step magnitude
Within each workbook, steps are grouped into **families that share a starting rate** (for example 5 → 2.5, 1, 0.5, 0.25, 0.1 s⁻¹). This holds the starting state fixed, so only the magnitude changes. For each family with at least three steps, the Spearman correlation ρ between |log₁₀ ratio| and each feature is computed. The ρ values are averaged per workbook, and a sign test across workbooks asks whether the relation has a consistent direction.

In [ ]:
rows = []
for (unit, g0, dirn), fam in A.groupby(["unit", "rate_from", "direction"]):
    if len(fam) < 3: continue
    for f in FEATURES:
        x, y = fam.log10_ratio.abs(), fam[f]
        ok = np.isfinite(y)
        if ok.sum() >= 3 and y[ok].nunique() > 1:
            rows.append(dict(unit=unit, sample=fam["sample"].iloc[0], family=f"{g0:g}->*", direction=dirn,
                             feature=f, n=ok.sum(), rho=stats.spearmanr(x[ok], y[ok])[0]))
mag = pd.DataFrame(rows)
per_wb = mag.groupby(["feature", "direction", "sample"]).rho.mean().reset_index()
res_mag = []
for (f, dirn), g in per_wb.groupby(["feature", "direction"]):
    p, k, n = sign_test(g.rho)
    res_mag.append(dict(feature=f, direction=dirn, n_workbooks=n, median_rho=g.rho.median(),
                        workbooks_rho_pos=k, sign_test_p=p, families=int((mag.feature.eq(f) & mag.direction.eq(dirn)).sum())))
res_mag = pd.DataFrame(res_mag).sort_values(["direction", "sign_test_p"])
res_mag.to_csv(OUT / "result_magnitude.csv", index=False)
res_mag.round(3)

In [ ]:
show = ["jump_signed", "init_slope", "settling_time", "area_log"]
fig, ax = plt.subplots(2, len(show), figsize=(13, 5.5))
for row, dirn in enumerate(["down", "up"]):
    for col, f in enumerate(show):
        a = ax[row, col]
        for i, ((unit, g0), fam) in enumerate(A[A.direction == dirn].groupby(["unit", "rate_from"])):
            if len(fam) < 3: continue
            fam = fam.sort_values("log10_ratio")
            a.plot(fam.log10_ratio.abs(), fam[f], "o-", ms=3, lw=0.8, alpha=0.6, color=f"C{i % 10}")
        a.set(title=f"step {dirn}: {f}", xlabel="|log10(final/initial rate)|")
fig.suptitle("Each line = one family (same workbook, same starting rate)", fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "05_magnitude.png", dpi=200); plt.show()

**Result (magnitude).** Magnitude is clearly encoded, but by different features for each direction:
* **Step-downs** (8–9 workbooks): a larger drop in rate gives a **deeper undershoot** (ρ = 1.0 in 8/8 workbooks), a **later minimum** `t_min` (9/9) and a larger **viscosity ratio** (8/8).
* **Step-ups** (7 workbooks): a larger rise gives a **larger overshoot** and **signed jump** (7/7), a **steeper initial decay** and **faster settling** (7/7).
* `area_log` is not consistent for either direction.

With 7–9 independent workbooks, p ≈ 0.004–0.016 is the smallest a sign test can give, so these are strong *within this dataset* but still exploratory.

### 7.2 Step direction — matched reverse pairs only
Comparing all step-ups with all step-downs would be confounded: step-downs almost always start at a high rate and step-ups at a low one, so a naive comparison partly detects the *starting rate*. The fair test uses **matched reverse pairs within one workbook**, i.e. a → b and b → a between the same two rates (5 ↔ 0.25, 5 ↔ 0.1, 10 ↔ 0.1, 20 ↔ 0.1). For each pair, the up-minus-down difference of each feature is computed, averaged per workbook, and tested across workbooks.

A physically motivated **no-training rule** is also evaluated: *predict “up” if `jump_signed` > 0* (stress starts above its new steady value and relaxes down). Its accuracy is reported with a 95 % CI from a bootstrap that resamples whole workbooks.

In [ ]:
pairs = []
for unit, g in A.groupby("unit"):
    for _, u in g[g.direction == "up"].iterrows():
        dn = g[(g.direction == "down") & np.isclose(g.rate_from, u.rate_to) & np.isclose(g.rate_to, u.rate_from)]
        if len(dn):
            dn = dn.iloc[0]
            pairs.append(dict(unit=unit, sample=u["sample"], pair=f"{u.rate_from:g}<->{u.rate_to:g}",
                              up_id=u.step_id, down_id=dn.step_id,
                              **{f"d_{f}": u[f] - dn[f] for f in FEATURES},
                              **{f"up_{f}": u[f] for f in FEATURES}, **{f"down_{f}": dn[f] for f in FEATURES}))
pairs = pd.DataFrame(pairs)
print(f"{len(pairs)} matched reverse pairs from {pairs['sample'].nunique()} workbooks:",
      pairs.groupby("pair").size().to_dict())

res_dir = []
for f in FEATURES:
    wbm = pairs.groupby("sample")[f"d_{f}"].mean()
    p, k, n = sign_test(wbm)
    w = stats.wilcoxon(wbm.dropna()).pvalue if wbm.notna().sum() >= 5 else np.nan
    res_dir.append(dict(feature=f, n_workbooks=n, median_up_minus_down=wbm.median(), workbooks_up_gt_down=k,
                        sign_test_p=p, wilcoxon_p=w, pairs_up_gt_down=f"{(pairs[f'd_{f}'] > 0).sum()}/{pairs[f'd_{f}'].notna().sum()}"))
res_dir = pd.DataFrame(res_dir).sort_values("sign_test_p")
res_dir.to_csv(OUT / "result_direction.csv", index=False)
res_dir.round(4)

In [ ]:
def rule_accuracy(df):
    ok = df.jump_signed.notna()
    return ((df.jump_signed[ok] > 0) == (df.direction[ok] == "up")).mean()

paired_ids = set(pairs.up_id) | set(pairs.down_id)
P = A[A.step_id.isin(paired_ids)]
rng = np.random.default_rng(0)
samples = P["sample"].unique()
boot = [rule_accuracy(pd.concat([P[P["sample"] == s] for s in rng.choice(samples, len(samples))])) for _ in range(2000)]
print(f"Direction rule 'jump_signed > 0 => up' on matched pairs: accuracy {rule_accuracy(P):.2f} "
      f"(95% workbook-bootstrap CI {np.percentile(boot, 2.5):.2f}-{np.percentile(boot, 97.5):.2f}, n = {len(P)} steps)")
print(f"Same rule on all steps (confounded by starting rate): {rule_accuracy(A):.2f}")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, f in zip(ax, ["jump_signed", "init_slope", "settling_time"]):
    for _, r in pairs.iterrows():
        a.plot([0, 1], [r[f"down_{f}"], r[f"up_{f}"]], "-o", ms=3, alpha=0.5)
    a.set_xticks([0, 1], ["step down", "step up"]); a.set_title(f"{f} (lines join matched pairs)")
fig.tight_layout(); fig.savefig(FIG / "06_direction_pairs.png", dpi=200); plt.show()

**Result (direction).** The **pre-specified rule failed**: `jump_signed > 0 ⇒ up` is right for only 55 % of matched steps (95 % CI 46–64 %), no better than chance. The reason is visible in §6.4: straight after a step down, the stress is still relaxing elastically from its previous, higher level, so the first point is often *above* the new plateau. This negative result is kept.

Among the other features, three separate up from down in the matched pairs very consistently:
* **undershoot**: larger for the step down in **22/22 pairs** (10/10 workbooks)
* **init_slope**: steeper decline for the step down in **22/22 pairs**
* **area_log**: larger for the step up in **21/22 pairs**

These were identified **after** seeing the data, so they are exploratory and need confirmation on new samples before being used as a classifier. They also make physical sense: a step down produces an elastic dip followed by structural recovery (the undershoot), while a step up produces an overshoot that decays within about 1 s, as Pincot et al. (2023) describe.

### 7.3 Transient vs stable portions
For every measurement, three window-level descriptors are computed separately in W_T and W_S. They use only the window's own data and σ_stable as a scale:

* `abs_slope`: |d(σ/σ_stable)/d log₁₀ t| — how fast stress is still changing
* `abs_dev`: mean |σ − σ_stable|/σ_stable — distance from the steady value
* `cv`: coefficient of variation of σ within the window

The comparison is **paired within each measurement** (W_T vs W_S of the same step), then summarised per workbook and tested across workbooks.

In [ ]:
def window_desc(d, t0, t1, ss):
    w = window(d, t0, t1)
    if len(w) < 3 or not np.isfinite(ss) or abs(ss) < NOISE_PA: return dict(abs_slope=np.nan, abs_dev=np.nan, cv=np.nan)
    x = np.log10(w.time.values); y = w.stress.values / ss
    return dict(abs_slope=abs(np.polyfit(x, y, 1)[0]), abs_dev=np.mean(np.abs(y - 1)), cv=np.std(y) / abs(np.mean(y)))

def phase_table(t_trans_end, t_stable_start):
    rows = []
    for _, r in A.iterrows():
        b = next(bb for bb in blocks if bb["step_id"] == r.step_id)
        ws = window(b["an"], t_stable_start, np.inf); ss = np.median(ws.stress) if len(ws) else np.nan
        rows.append(dict(step_id=r.step_id, sample=r["sample"], stable_ok=r.stable_ok,
                         **{f"T_{k}": v for k, v in window_desc(b["an"], r.t_rate_settle, t_trans_end, ss).items()},
                         **{f"S_{k}": v for k, v in window_desc(b["an"], t_stable_start, np.inf, ss).items()}))
    return pd.DataFrame(rows)

def phase_result(ph, label):
    out = []
    for k in ["abs_slope", "abs_dev", "cv"]:
        diff = np.log10(ph[f"T_{k}"].clip(lower=1e-6) / ph[f"S_{k}"].clip(lower=1e-6))   # log ratio transient/stable
        wbm = diff.groupby(ph["sample"]).median()
        p, kk, n = sign_test(wbm)
        out.append(dict(rule=label, descriptor=k, measurements_T_gt_S=f"{(diff > 0).sum()}/{diff.notna().sum()}",
                        median_log10_ratio_T_over_S=diff.median(), workbooks_T_gt_S=f"{kk}/{n}", sign_test_p=p))
    return out

ph = phase_table(TRANSIENT_END, STABLE_START)
res_phase = pd.DataFrame(phase_result(ph, "primary (2 s / 30 s)"))
res_phase.round(4)

### 7.4 Sensitivity to the transient/stable boundaries
The framework asks for the boundary choice to be checked. The same test is repeated with shorter and longer windows. A conclusion that holds under all three is robust; one that flips is reported as boundary-dependent.

In [ ]:
sens = pd.concat([pd.DataFrame(phase_result(phase_table(a, b), name)) for name, (a, b) in RULE_VARIANTS.items()])
sens.to_csv(OUT / "result_transient_vs_stable.csv", index=False)
sens.pivot(index="descriptor", columns="rule", values="workbooks_T_gt_S")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.3))
for a, k in zip(ax, ["abs_slope", "abs_dev", "cv"]):
    a.boxplot([ph[f"T_{k}"].dropna(), ph[f"S_{k}"].dropna()], tick_labels=["transient W_T", "stable W_S"], showfliers=False)
    for _, r in ph.iterrows():
        a.plot([1, 2], [r[f"T_{k}"], r[f"S_{k}"]], color="grey", alpha=0.15, lw=0.6)
    a.set_yscale("log"); a.set_title(k)
fig.tight_layout(); fig.savefig(FIG / "07_transient_vs_stable.png", dpi=200); plt.show()

**Result (transient vs stable).** All three descriptors are higher in the transient window than in the stable window **in 11/11 workbooks** (127/128 measurements for `abs_dev` and `cv`). The conclusion holds under all three boundary choices. The one exception is `abs_slope` with the long rule (5 s / 40 s), which drops to 9/11 because by 5 s most responses have already levelled off. **`abs_dev` and `cv` are the robust choice; `abs_slope` depends on where the transient window ends.**

### 7.5 Storage time (DEC17 at 0, 12 and 24 h)
Not one of the four framework questions, but DEC17 is the only workbook where the **same blood** was stepped at three storage times, which makes it the clearest demonstration of change. These are repeated measurements of one sample, so this is descriptive only.

In [ ]:
dec = A[A["sample"] == "DEC17"]
if len(dec):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.3))
    for a, f in zip(ax, ["sigma_stable", "jump_signed", "tau_eff"]):
        for i, (lab, g) in enumerate(dec.groupby("label")):
            g = g.sort_values("storage_h")
            if g.storage_h.nunique() == 3 and lab.split("(")[0].strip() in ("5-->.1", ".1-->5", "5-->1", ".25-->1", ".25-->5", "5-->2.5"):
                a.plot(g.storage_h, g[f], "o-", label=lab.split("(")[0], color=f"C{i % 10}")
        a.set(xlabel="storage time (h)", title=f, xticks=[0, 12, 24])
        if f == "sigma_stable": a.set_yscale("log")
    ax[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG / "08_dec17_storage.png", dpi=200); plt.show()
    display(models[models["sample"] == "DEC17"][["storage_h", "tau_lambda", "G", "sigma_y0", "eta_ST", "eta_inf", "tr1", "tr2"]].round(4))

**Result (DEC17 storage).** In the same blood over 24 h: σ_stable rises with storage time for all six step conditions shown, and the effective relaxation time τ_eff falls; the model gives **G 0.22 → 0.30 → 0.46 Pa** and **η_ST 0.021 → 0.036 → 0.34 Pa·s** (about 16× by 24 h), while τ_λ falls from 2.0 to about 1.1 s. This is consistent with increasing aggregation and stiffening of stored blood. It is one sample measured three times, so it is descriptive only.

## 8. Which transient metrics are reliable, inconclusive or unusable?

Each feature is classified with explicit, pre-stated criteria:

* **Computable fraction**: share of analysable measurements where the feature is finite.
* **Stable-state dependence**: features divided by σ_stable are downgraded when many measurements lack a defensible stable state.
* **Per question** (magnitude of step-downs, magnitude of step-ups, direction): `yes` if the effect has the same sign in ≥ 80 % of workbooks **and** the sign test gives p < 0.05, otherwise `no`.

| Class | Rule |
|---|---|
| reliable | computable in ≥ 80 % of measurements **and** `yes` for at least one question |
| inconclusive | computable in ≥ 50 % but no consistent effect |
| unusable | computable in < 50 % of measurements, or fitting fails in most cases |

With about 10 workbooks, even “reliable” means *consistent in this exploratory dataset*, not confirmed.

In [ ]:
def verdict(p, share):
    """'yes' = consistent in >= 80 % of workbooks AND sign-test p < 0.05; otherwise 'no'."""
    if not np.isfinite(p): return "n/a"
    return "yes" if p < 0.05 and max(share, 1 - share) >= 0.8 else "no"

rel = []
for f in FEATURES:
    row = dict(feature=f, computable=round(A[f].notna().mean(), 2))
    for dirn in ("down", "up"):
        m = res_mag[(res_mag.feature == f) & (res_mag.direction == dirn)]
        row[f"magnitude_{dirn}"] = (verdict(m.sign_test_p.iloc[0], m.workbooks_rho_pos.iloc[0] / max(m.n_workbooks.iloc[0], 1))
                                    if len(m) else "n/a")
    d = res_dir[res_dir.feature == f]
    row["direction"] = verdict(d.sign_test_p.iloc[0], d.workbooks_up_gt_down.iloc[0] / max(d.n_workbooks.iloc[0], 1)) if len(d) else "n/a"
    n_yes = sum(row[k] == "yes" for k in ("magnitude_down", "magnitude_up", "direction"))
    row["needs_stable_state"] = f in ("jump_signed", "overshoot", "undershoot", "init_slope", "area_log", "settling_time")
    row["classification"] = "unusable" if row["computable"] < 0.5 else "reliable" if row["computable"] >= 0.8 and n_yes else "inconclusive"
    rel.append(row)
fit_share = (fits_exp.status == "ok").mean()
rel.append(dict(feature="tau (single exponential)", computable=round(fit_share, 2),
                classification="unusable" if fit_share < 0.5 else "inconclusive"))
rel.append(dict(feature="tau_lambda, G (ethixo-EVP, per workbook)", computable=round((~models.k_at_bound).mean(), 2),
                classification="reliable" if (~models.k_at_bound).mean() >= 0.8 else "inconclusive"))
rel = pd.DataFrame(rel)
rel.to_csv(OUT / "metric_reliability.csv", index=False)
rel

## 9. Summary for the report and hand-over to Members 4 and 5

In [ ]:
print("=== Data and verification ===")
print(f"step blocks analysed: {len(A)} from {A['sample'].nunique()} workbooks; label mismatches excluded: {(steps.qc_rate == 'MISMATCH with label').sum()}")
print(f"instrument window (rate settling): median {ev.t_rate_settle.median():.3f} s, max {ev.t_rate_settle.max():.3f} s")
print(f"defensible stable state within 63 s: {A.stable_ok.mean():.0%} of steps")
print("\n=== Relaxation model ===")
print(f"single exponential interpretable: {(fits_exp.status == 'ok').mean():.0%} of steps")
okm = models[~models.k_at_bound]
print(f"ethixo-EVP tau_lambda: median {okm.tau_lambda.median():.2f} s over {len(okm)} identifiable fits "
      f"(literature 1.13 +- 0.24 s); steps with acceptable fit (NRMSE <= {MODEL_MAX_NRMSE}): "
      f"{(A.nrmse_model <= A.fit_threshold).mean():.0%}")
print("\n=== Questions ===")
print("Magnitude (strongest):"); print(res_mag.nsmallest(3, "sign_test_p")[["feature", "direction", "median_rho", "workbooks_rho_pos", "n_workbooks", "sign_test_p"]].round(3).to_string(index=False))
print("Direction (matched pairs):"); print(res_dir.head(3)[["feature", "median_up_minus_down", "workbooks_up_gt_down", "n_workbooks", "sign_test_p"]].round(4).to_string(index=False))
print("Transient vs stable (all rule variants):"); print(sens[["rule", "descriptor", "workbooks_T_gt_S", "sign_test_p"]].round(4).to_string(index=False))
print(f"\nAll tables written to {OUT.resolve()}")

### Notes for the rest of the team

* **Member 1 (pipeline):** the reader in §1 and the event table in §3 can be moved into the shared pipeline. Step blocks *do* contain 3–6 pre-step samples; the instrument window must be excluded; high-resolution blocks need log-binning to be comparable.
* **Member 4 (transforms):** use the same `step_id`s, the same instrument window and the same W_T / W_S windows from `step_windows.csv`. Only the six high-resolution 5 → 0.25 s⁻¹ blocks support CWT/DWT on the raw series, and they are noise-dominated at low stress; compare transform features against the conventional baselines here (`jump_signed`, `init_slope`, `settling_time`, `tau_lambda`).
* **Member 5 (validation):** the per-workbook summaries in `result_*.csv` are already at the independent-unit level. Direction must be evaluated on matched reverse pairs only. Steps to 0.1–0.25 s⁻¹ often never stabilise within 63 s; treat their normalised features with caution.
* **Haemograph relevance (preliminary):** the cheapest candidate transient features are `undershoot`, `init_slope` and `settling_time` (one minimum, one regression over the first second, one threshold crossing). `tau_lambda` and `G` need a flow curve and an ODE fit, so they are better suited to offline characterisation.

### Limitations
Around 10 independent blood samples; results are exploratory. The model is the scalar (non-tensorial) ethixo-EVP version 1 with n = 1; the papers' later tensorial models (t-e-TVP, t-ESSTV) were not implemented. k_λ's interval is a conditional profile (other parameters held fixed), so it understates the true uncertainty. The pre-step state is assumed to be steady at the initial rate. Changes in stress are *consistent with* rouleaux breakdown and aggregation but do not prove a mechanism.